In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
JOB_INDEX = 0  # 0..17 -- which manifest entry to run, one per session

INPUT_DIR = ""  # "" = auto-detect the retrain_lora_input dataset under /kaggle/input
HF_TOKEN = ""  # "" = fall back to Kaggle Secret HF_TOKEN

PARALLEL = True  # True | False

SPLIT_BUDGETS = True  # True | False

OUTPUT_DIR = "/kaggle/working/checkpoints"

MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

# No FEATURE_DIR / CELLVIT_DIR / VLM_FEATURE_DIR here, unlike run_al_main:
#   CellViT feeds SELECTION only, and selection is what this notebook reads
#     from disk instead of recomputing.
#   The CONCH cache holds FROZEN-encoder features. Every job here trains the
#     encoder, so those features no longer describe it -- finetune_and_evaluate
#     re-encodes the test set through the ADAPTED encoder, which is the only
#     space its probe was ever trained to read.
#   DINOv2's cache belongs to a different encoder entirely.
# The image dataset IS required: this pass reads pixels.


In [ ]:
import json

if INPUT_DIR:
    IN = Path(INPUT_DIR)
else:
    roots = sorted(Path("/kaggle/input").glob("*/manifest.json"))
    assert roots, "no manifest.json under /kaggle/input -- attach retrain_lora_input.zip"
    IN = roots[0].parent
MANIFEST = json.loads((IN / "manifest.json").read_text())
assert 0 <= JOB_INDEX < len(MANIFEST), f"JOB_INDEX must be 0..{len(MANIFEST) - 1}"

print(f"input: {IN}  |  {len(MANIFEST)} job")
for i, m in enumerate(MANIFEST):
    mark = ">>" if i == JOB_INDEX else "  "
    tag = "" if m["already_exists"] else "  <- run does not exist yet"
    print(f"{mark} [{i:>2}] {m['code']}  {m['dataset']:<11} seed{m['seed']:<4}{tag}")

JOB = MANIFEST[JOB_INDEX]
DATASET = JOB["dataset"]
SEED = JOB["seed"]
RUN = JOB["run_name"]
FINAL_TRAIN_CFG = JOB["final_train_cfg"]
BUDGETS = JOB["budgets"]

# T3 and T4 differ ONLY in `augment`, which acts inside the final-training
# pass: their selections are index-identical at all 8 budgets (verified on
# every cell that has both). So one saved selection drives both rows, which is
# why 9 selection sets cover 18 jobs -- and why 7 of these jobs are runs that
# never existed rather than recoveries.
print()
print("job:", JOB["code"], DATASET, "seed", SEED)
print("run_name:", RUN)
print("final_train_cfg:", FINAL_TRAIN_CFG)
print("selection from:", JOB["sel_dir"])

In [ ]:
import os

# CONCH is gated and this pass always loads it: every job here is a LoRA run,
# so unlike run_al_main there is no cache-only branch to fall through to.
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("[auth] using Kaggle Secret HF_TOKEN")
    except Exception as exc:
        print(f"[auth] no Kaggle Secret HF_TOKEN ({type(exc).__name__})")
assert HF_TOKEN, (
    "This notebook loads the GATED CONCH checkpoint for every job. Set "
    "HF_TOKEN in the EDIT cell, or add a Kaggle Secret named HF_TOKEN "
    "(Add-ons -> Secrets). Without one the download fails with a 401/403 "
    "partway into the run."
)
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login

login(HF_TOKEN)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "git+https://github.com/mahmoodlab/CONCH.git"])
print("conch installed")

In [ ]:
import yaml
import torch

import main
from training.retrain import retrain_on_worker
from utils.parallel import run_variants_parallel, visible_gpu_count
from utils.kaggle import find_data_root
from utils.progress import format_duration

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]

# The selection this pass reuses was made with these; they are recorded so the
# archive says which table row it is without anyone parsing the filename.
SAMPLER_CFG = {**config.get("samplers", {}).get("pact", {}), **JOB["sampler_cfg"]}

DATA_ROOT = find_data_root()
DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running"

# The backbone comes from the manifest, never re-chosen here: a re-train that
# swapped encoders would hand the paper an adapter for a different model.
VISUAL_BACKBONE = JOB["visual_backbone"]
IMAGE_ENCODER = "conch" if "CONCH" in VISUAL_BACKBONE else "dinov2"
assert FINAL_TRAIN_CFG["use_lora"], (
    "every job in this notebook is a LoRA run -- a frozen run has no adapter "
    "to recover and its probe is already reproducible from the feature cache"
)

SEL_DIR = IN / JOB["sel_dir"]
assert SEL_DIR.is_dir(), f"missing {SEL_DIR}"
# Two naming schemes: an archive's own `*_selected_budget_<b>.pt`, or the
# input dataset's shortened `budget_<b>.pt`. The zip uses the short one --
# Kaggle rejects any entry over 248 bytes, and the full run name appearing
# twice in one path (directory + filename) reached 254.
found = sorted(SEL_DIR.glob("*_selected_budget_*.pt")) or sorted(SEL_DIR.glob("budget_*.pt"))
assert len(found) == len(BUDGETS), (
    f"{SEL_DIR}: {len(found)} selection files for {len(BUDGETS)} budgets"
)

SAVE_DIR = Path(OUTPUT_DIR) / DATASET
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("dataset:", DATASET, "| seed:", SEED, "| encoder:", IMAGE_ENCODER)
print("selection files:", len(found))
print("budgets:", BUDGETS)
print("GPUs visible:", visible_gpu_count())

In [ ]:
import time

workers = visible_gpu_count() if PARALLEL else 1
shard_budgets = SPLIT_BUDGETS and workers > 1 and len(BUDGETS) > 1

def budget_shards(budgets, n):
    """Round-robin, not contiguous: cost grows with the budget, so a contiguous
    split would hand one worker every large budget."""
    groups = [budgets[i::n] for i in range(n)]
    return [g for g in groups if g]

base_kwargs = dict(
    run_dir=str(SEL_DIR),
    data_path=str(data_path),
    save_dir=str(SAVE_DIR),
    run_name=RUN,
    final_train_cfg=FINAL_TRAIN_CFG,
    num_classes=dataset_info["num_classes"],
    dataset_key=DATASET,
    sampler_name="pact",
    # Not used to select -- carried into `_results.pt` so the recovered archive
    # is classifiable by CONFIG (uncertainty_mode + pool_consistency_weight is
    # what names the table row) rather than by parsing its filename.
    sampler_cfg=SAMPLER_CFG,
    random_seed=SEED,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    image_encoder=IMAGE_ENCODER,
    visual_backbone=VISUAL_BACKBONE,
    hf_token=HF_TOKEN or None,
    mmap_cache_dir=MMAP_CACHE_DIR,
    device_string="cuda:0",
    verbose=True,
)

jobs = []
shard_tags = []
if (SAVE_DIR / f"{RUN}_results.pt").is_file():
    print(f"already finished, nothing to do: {RUN}_results.pt")
elif shard_budgets:
    shards = budget_shards(BUDGETS, workers)
    shard_tags = [f"shard{i}" for i in range(len(shards))]
    for tag, budgets in zip(shard_tags, shards):
        jobs.append((f"{RUN}:{tag}", dict(
            base_kwargs, budgets=budgets, shard_tag=tag,
        )))
else:
    jobs.append((RUN, dict(base_kwargs, budgets=BUDGETS)))

print(f"run: {RUN}")
for label, kwargs in jobs:
    print(f"   {label[-46:]:46} budgets={kwargs['budgets']}")

if str(data_path).endswith(".npz"):
    from data.npz_mmap import export_npz_to_npy

    export_npz_to_npy(str(data_path), MMAP_CACHE_DIR)
    print(f"mmap export ready: {MMAP_CACHE_DIR}")
else:
    print("ImageFolder dataset: reads per file, no mmap export needed")

started = time.time()
results = run_variants_parallel(
    jobs, retrain_on_worker, num_workers=workers,
    abort_on_failure=bool(shard_tags),
)

print("=" * 70)
for result in results:
    status = "ok" if result["ok"] else "FAILED"
    print(f"{result['label'][-46:]:46} {status:8} {format_duration(result['seconds'])}")
failed = [r["label"] for r in results if not r["ok"]]
if results:
    print(f"total {format_duration(time.time() - started)} | "
          f"{len(results) - len(failed)}/{len(results)} succeeded")
for result in results:
    if not result["ok"]:
        print("=" * 70)
        print(f"TRACEBACK -- {result['label']}")
        print(result["error"])
assert not failed, f"jobs failed: {failed}"

if shard_tags:
    linear = main.merge_budget_shards(str(SAVE_DIR), RUN, shard_tags)
    print(f"[merge] {RUN}: {len(linear)} budgets -> {RUN}_results.pt")

In [ ]:
import torch

results_path = SAVE_DIR / f"{RUN}_results.pt"
assert results_path.is_file(), (
    f"no results at {results_path} -- the run above did not finish"
)
payload = torch.load(results_path, weights_only=False)
linear = payload["linear"]

print(f"{payload['sampler']}  |  {payload['dataset']}  |  seed {payload['seed']}")
print(f"run_name: {payload['run_name']}")
if payload.get("sharded_over"):
    print(f"budget shards: {', '.join(payload['sharded_over'])}")
print()

header = f"{'budget':>8}  {'accuracy':>9}  {'precision':>9}  {'recall':>9}  {'macro F1':>9}"
print(header)
print("-" * len(header))
for budget in sorted(linear):
    row = linear[budget]
    print(f"{budget:>8}  {row['acc']:>9.4f}  {row['precision']:>9.4f}  "
          f"{row['recall']:>9.4f}  {row['f1']:>9.4f}")
print("-" * len(header))
best = max(linear, key=lambda b: linear[b]["acc"])
print(f"best accuracy {linear[best]['acc']:.4f} at budget {best}")

# Every budget must have left an adapter behind: that is the entire reason
# this notebook exists, and a run that produced none would look like a success
# in the table above.
missing = [b for b in BUDGETS if not (SAVE_DIR / f"{RUN}_lora_budget_{b}.pt").is_file()]
assert not missing, f"no adapter written for budgets {missing}"
size = sum(f.stat().st_size for f in SAVE_DIR.glob(f"{RUN}_lora_budget_*.pt"))
print(f"\nadapters: {len(BUDGETS)} files, {size / 1e6:.1f} MB")

sample = torch.load(
    SAVE_DIR / f"{RUN}_lora_budget_{BUDGETS[0]}.pt",
    map_location="cpu", weights_only=False,
)
assert sample["state"], "adapter file holds an empty state dict"
nonzero = sum(float(t.abs().sum()) for k, t in sample["state"].items() if "lora_B" in k)
assert nonzero > 0, (
    "every lora_B is zero -- this adapter reconstructs the FROZEN encoder, so "
    "it was captured before training rather than after"
)
print(f"r={sample['lora_r']} alpha={sample['lora_alpha']} tensors={len(sample['state'])}")

In [ ]:
import shutil

from utils import main_archive_stem

if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")

SOURCE = Path(OUTPUT_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "OUTPUT_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = main_archive_stem(DATASET, "pact", SEED, encoder=IMAGE_ENCODER, run_name=RUN)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.rglob("*")):
    if path.is_file():
        print(f"    {path.relative_to(SOURCE)}  ({path.stat().st_size / 1e6:.2f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)
print(f"\nNEXT: Output tab -> download the zip, then set JOB_INDEX to the next job.")